In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

: 

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("No GPU detected, using CPU.")

# device = torch.device("cpu")

In [ ]:
def init_cnn(module):  #@save
    """Initialize weights for CNNs."""
    if type(module) == nn.Linear or type(module) == nn.Conv2d:
        nn.init.xavier_uniform_(module.weight)

class LeNet(d2l.Classifier):  #@save
    """The LeNet-5 model."""
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.LazyConv2d(6, kernel_size=5, padding=2), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.LazyConv2d(16, kernel_size=5), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.LazyLinear(120), nn.Sigmoid(),
            nn.LazyLinear(84), nn.Sigmoid(),
            nn.LazyLinear(num_classes))

In [ ]:
@d2l.add_to_class(d2l.Classifier)  #@save
def layer_summary(self, X_shape):
    X = torch.randn(*X_shape)
    for layer in self.net:
        X = layer(X)
        print(layer.__class__.__name__, 'output shape:\t', X.shape)

model = LeNet()
model.layer_summary((1, 1, 28, 28))

In [ ]:
from torch.utils.data import DataLoader

if __name__ == '__main__':
    model = LeNet()
    model.layer_summary((1, 1, 28, 28))
    
    # Create the trainer (using one GPU if available)
    trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
    
    data = d2l.FashionMNIST(batch_size=128)
    
    # Since data.get_dataloader() does not accept a num_workers argument,
    # we manually create a DataLoader for the training set with num_workers=0.
    # The d2l.FashionMNIST object typically has attributes 'train' and 'test'.
    train_dataset = data.train
    train_loader = DataLoader(train_dataset, batch_size=data.batch_size, shuffle=True, num_workers=0) 
    # what is num_workers? when using torch cpu it doesnt crash but installing cuda makes this crash without making a custom dataloader with num_workers=0
    
    # Retrieve one batch from our custom train_loader for initialization.
    try:
        sample = next(iter(train_loader))[0]
    except Exception as e:
        print("Error while fetching a batch from the DataLoader:", e)
        raise e

    # Reinitialize the model with the desired learning rate and apply initialization.
    model = LeNet(lr=0.1)
    model.apply_init([sample], init_cnn)
    
    # Fit the model using the trainer and the data object.
    trainer.fit(model, data)

# trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
# data = d2l.FashionMNIST(batch_size=64)
# model = LeNet(lr=0.1)
# model.apply_init([next(iter(data.get_dataloader(True)))[0]], init_cnn)
# trainer.fit(model, data)